# Strategic Investment Banking OS — Transparent Manual Loop 003

This notebook adds an evidence-provenance layer to the synthetic VoltEdge opportunity:

1. locate and snapshot the vault;
2. define a controlled synthetic source packet;
3. register material claims separately from sources;
4. distinguish reported facts, calculations, assumptions and analyst inference;
5. calculate every evidence-quality component;
6. test staleness and corroboration;
7. detect and preserve a capacity-utilization contradiction;
8. apply a governance decision gate;
9. preview every proposed mutation;
10. write only after explicit human authorization.

No real source, URL, company or market fact is used. No LLM or external API is called.


## The principle being tested

An institutional second brain must remember not only a conclusion, but also **who said what, when, in which document, how the claim was calculated, what contradicts it and when it must be reviewed**. Evidence should sometimes weaken or block an opportunity rather than automatically confirm it.


In [ ]:
# 1. Imports and safety configuration

from pathlib import Path
from datetime import date, datetime, timezone
from html import escape
from collections import Counter
import csv, hashlib, io, json, os, shutil, tempfile

from IPython.display import display, HTML, Markdown

VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"
LOOP_ID = "LOOP-003-COLAB"
AS_OF = date(2026, 7, 17)

COMMIT_CHANGES = False
BACKUP_BEFORE_WRITE = True

print("Loop:", LOOP_ID)
print("As of:", AS_OF)
print("COMMIT_CHANGES:", COMMIT_CHANGES)


In [ ]:
# 2. Mount Drive and resolve the independent vault

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
else:
    MY_DRIVE = Path.cwd()

VAULT = MY_DRIVE / VAULT_NAME
if not VAULT.exists():
    matches = list(MY_DRIVE.glob(f"**/{VAULT_NAME}"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one vault; found {matches}")
    VAULT = matches[0]

print("Vault:", VAULT)
print("VoltEdge note present:", (VAULT / "Companies/VoltEdge Thermal Systems.md").exists())


In [ ]:
# 3. Snapshot institutional memory before analysis

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files_before = sorted(p for p in VAULT.rglob("*") if p.is_file())
manifest_before = {str(p.relative_to(VAULT)): {"bytes": p.stat().st_size, "sha256": sha256(p)} for p in files_before}

with (VAULT / "Data/company_master.csv").open(encoding="utf-8", newline="") as f:
    companies = list(csv.DictReader(f))
voltedge = next(r for r in companies if r["id"] == "SYN-102")

print("Files before run:", len(files_before))
print("Companies:", len(companies))
print("Evidence target:", voltedge["name"])


In [ ]:
# 4. Transparent display helper

def display_table(rows, columns, title=None):
    if title: display(Markdown(f"### {title}"))
    head = "".join(f"<th style='padding:6px'>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td style='padding:6px;border-top:1px solid #ddd'>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>" for r in rows)
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))


## Controlled synthetic source packet

Four sources deliberately have different authority and independence. Internal management material can be direct and recent without being independent. An external interview can be more independent while still being incomplete. A market outlook remains a scenario input—not an observed fact.


In [ ]:
# 5. Register sources with explicit provenance and limitations

SOURCES = [
    {"id":"SRC-001","title":"VoltEdge Management Presentation Q2 2026","date":"2026-07-10","type":"Management presentation","author":"VoltEdge management (fictional)","authority":24,"independence":0,"recency":15,"summary":"Revenue $147m; growth 40%; recurring revenue 45%; nominal capacity utilization 92%.","limitation":"Unaudited and not independent."},
    {"id":"SRC-002","title":"VoltEdge Synthetic Financial Schedule FY2025","date":"2026-07-08","type":"Financial schedule","author":"Synthetic finance function","authority":36,"independence":0,"recency":15,"summary":"Prior revenue $105m; revenue $147m; EBITDA $14.7m; FCF -$6.5m; customer concentration 38%.","limitation":"No audit opinion or underlying ledger."},
    {"id":"SRC-003","title":"VoltEdge Synthetic Operations Diligence Interview","date":"2026-07-11","type":"Operations interview","author":"Independent diligence consultant (fictional)","authority":26,"independence":15,"recency":15,"summary":"Effective capacity utilization estimated at 78% after downtime and changeovers.","limitation":"Single interview; not reconciled to plant records."},
    {"id":"SRC-004","title":"Synthetic AI Data Center Market Outlook 2026","date":"2026-07-05","type":"Market outlook","author":"Independent research provider (fictional)","authority":28,"independence":16,"recency":15,"summary":"Scenario assumes data-center capex rises 30%; cooling, power and interconnection constrain growth.","limitation":"Scenario input, not observed market evidence."},
]

assert len({s["id"] for s in SOURCES}) == len(SOURCES)
display_table(SOURCES, ["id","title","date","type","authority","independence","recency","limitation"], "Source registry")


## Claim taxonomy

- **Reported fact:** a source states the value directly.
- **Derived calculation:** the notebook calculates it from disclosed inputs.
- **Disputed fact:** sources disagree or use incompatible definitions.
- **Scenario assumption:** an analytical input, not a statement of reality.
- **Analyst inference:** professional interpretation; confidence is capped at 70.


In [ ]:
# 6. Register claims separately from the documents

CLAIMS = [
    {"id":"CLM-001","statement":"VoltEdge FY2025 revenue is $147.0m.","claim_type":"reported-fact","sources":["SRC-001","SRC-002"],"authority":36,"independence":0,"recency":15,"corroboration":15,"directness":10,"penalty":0,"status":"Corroborated","review_by":"2027-01-17"},
    {"id":"CLM-002","statement":"Revenue grew 40.0% from $105.0m to $147.0m.","claim_type":"derived-calculation","sources":["SRC-001","SRC-002"],"authority":36,"independence":0,"recency":15,"corroboration":15,"directness":6,"penalty":0,"status":"Recalculated","review_by":"2027-01-17"},
    {"id":"CLM-003","statement":"The top customer represents 38% of revenue.","claim_type":"reported-fact","sources":["SRC-002"],"authority":36,"independence":0,"recency":15,"corroboration":0,"directness":10,"penalty":0,"status":"Single-source","review_by":"2026-10-17"},
    {"id":"CLM-004","statement":"Current production-capacity utilization is between 78% and 92%.","claim_type":"disputed-fact","sources":["SRC-001","SRC-003"],"authority":26,"independence":15,"recency":15,"corroboration":0,"directness":10,"penalty":25,"status":"Contradiction open","review_by":"2026-08-17"},
    {"id":"CLM-005","statement":"The scenario assumes data-center capital expenditure rises 30%.","claim_type":"scenario-assumption","sources":["SRC-004"],"authority":28,"independence":16,"recency":15,"corroboration":0,"directness":8,"penalty":0,"status":"Assumption — not fact","review_by":"2026-10-17"},
    {"id":"CLM-006","statement":"Growth capital may be appropriate for capacity and working capital.","claim_type":"analyst-inference","sources":["SRC-001","SRC-002","SRC-004"],"authority":36,"independence":16,"recency":15,"corroboration":15,"directness":5,"penalty":0,"status":"Inference — human review","review_by":"2026-08-17"},
]

known_sources = {s["id"] for s in SOURCES}
assert all(set(c["sources"]) <= known_sources for c in CLAIMS)
assert len({c["id"] for c in CLAIMS}) == len(CLAIMS)
print("Claims registered:", len(CLAIMS))


In [ ]:
# 7. Calculate every confidence component, inference cap and staleness flag

def score_claim(c):
    raw = c["authority"] + c["independence"] + c["recency"] + c["corroboration"] + c["directness"] - c["penalty"]
    confidence = max(0, min(100, raw))
    if c["claim_type"] == "analyst-inference":
        confidence = min(70, confidence)
    return {**c, "raw_score": raw, "confidence_score": confidence,
            "stale": AS_OF > date.fromisoformat(c["review_by"])}

scored_claims = [score_claim(c) for c in CLAIMS]
display_table(scored_claims, ["id","claim_type","authority","independence","recency","corroboration","directness","penalty","raw_score","confidence_score","stale","status"], "Evidence-quality decomposition")

average_confidence = round(sum(c["confidence_score"] for c in scored_claims) / len(scored_claims), 1)
print("Average confidence:", average_confidence)


In [ ]:
# 8. Validate the derived financial claim independently

prior_revenue = float(voltedge["revenue_prev_usd_m"])
current_revenue = float(voltedge["revenue_usd_m"])
recalculated_growth = round((current_revenue / prior_revenue - 1) * 100, 1)
stored_growth = float(voltedge["revenue_growth_pct"])

calculation_check = {
    "prior_revenue": prior_revenue,
    "current_revenue": current_revenue,
    "recalculated_growth_pct": recalculated_growth,
    "stored_growth_pct": stored_growth,
    "matches": recalculated_growth == stored_growth == 40.0,
}
print(json.dumps(calculation_check, indent=2))
assert calculation_check["matches"]


In [ ]:
# 9. Detect rather than average away the controlled contradiction

capacity_observations = [
    {"source":"SRC-001","value_pct":92,"basis":"nominal rated capacity"},
    {"source":"SRC-003","value_pct":78,"basis":"effective capacity after downtime"},
]
spread = max(x["value_pct"] for x in capacity_observations) - min(x["value_pct"] for x in capacity_observations)
contradiction_open = spread >= 10 or len({x["basis"] for x in capacity_observations}) > 1

display_table(capacity_observations, ["source","value_pct","basis"], "Capacity observations")
print("Spread:", spread, "percentage points")
print("Contradiction open:", contradiction_open)
assert contradiction_open


In [ ]:
# 10. Apply an explicit governance decision rule

stale_claims = [c["id"] for c in scored_claims if c["stale"]]
low_confidence_claims = [c["id"] for c in scored_claims if c["confidence_score"] < 40]

if contradiction_open:
    decision = "CONDITIONAL — internal preliminary diligence only"
    prohibited = ["capacity-dependent valuation", "external outreach", "mandate approval"]
elif low_confidence_claims or stale_claims:
    decision = "HOLD — refresh evidence"
    prohibited = ["valuation reliance", "external outreach", "mandate approval"]
else:
    decision = "ELIGIBLE FOR HUMAN PRELIMINARY REVIEW"
    prohibited = ["automatic external action"]

decision_record = {"average_confidence":average_confidence,"open_contradictions":int(contradiction_open),"stale_claims":stale_claims,"low_confidence_claims":low_confidence_claims,"decision":decision,"prohibited":prohibited}
print(json.dumps(decision_record, indent=2))


In [ ]:
# 11. Construct source, claim, evidence and audit artifacts in memory

def csv_text(rows, fields):
    buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=fields, extrasaction="ignore")
    w.writeheader(); w.writerows(rows); return buf.getvalue()

source_fields = ["id","title","date","type","author","authority","independence","recency","summary","limitation"]
claim_fields = ["id","statement","claim_type","sources","authority","independence","recency","corroboration","directness","penalty","raw_score","confidence_score","status","review_by","stale"]
claim_csv_rows = [{**c,"sources":";".join(c["sources"])} for c in scored_claims]

source_rows = "\n".join(f"| {s['id']} | {s['title']} | {s['type']} | {s['date']} |" for s in SOURCES)
claim_rows = "\n".join(f"| {c['id']} | {c['claim_type']} | {c['status']} | {c['confidence_score']}/100 |" for c in scored_claims)

evidence_md = f'''---
type: evidence-pack-reproduction
loop_id: {LOOP_ID}
synthetic: true
---
# VoltEdge Evidence Pack — Colab Reproduction

| Claim | Classification | Status | Confidence |
|---|---|---|---:|
{claim_rows}

Average confidence: **{average_confidence}/100**.

Decision: **{decision}**. Capacity utilization remains disputed at 78% versus 92%.
'''
contradiction_md = f'''---
type: contradiction-register-reproduction
loop_id: {LOOP_ID}
open_items: 1
---
# Contradiction Register — Colab Reproduction

> [!contradiction] CON-001 — capacity utilization
> SRC-001 reports 92% nominal utilization. SRC-003 estimates 78% effective utilization. Spread: {spread} percentage points. Reconcile the denominator and plant records before reliance.
'''
audit_md = f'''---
type: audit-record
audit_id: AUD-003N
loop_id: {LOOP_ID}
---
# AUD-003N — Colab Provenance Reproduction

- Sources: {len(SOURCES)}
- Claims: {len(scored_claims)}
- Average confidence: {average_confidence}/100
- Open contradictions: {int(contradiction_open)}
- Stale claims: {len(stale_claims)}
- Real sources: 0
- LLM calls: 0
- External actions: 0
- Decision: {decision}
'''

planned_writes = {
    Path("Data/loop_003_colab_source_registry.csv"): csv_text(SOURCES, source_fields),
    Path("Data/loop_003_colab_claim_register.csv"): csv_text(claim_csv_rows, claim_fields),
    Path("Evidence/VoltEdge Evidence Pack - Colab Reproduction.md"): evidence_md,
    Path("Evidence/Contradiction Register - Colab Reproduction.md"): contradiction_md,
    Path("Audit/AUD-003N - Colab Provenance Reproduction.md"): audit_md,
}

for s in SOURCES:
    planned_writes[Path(f"Sources/{s['id']} - Colab Reproduction.md")] = f'''---
type: source-reproduction
source_id: {s['id']}
source_date: {s['date']}
synthetic: true
---
# {s['id']} — {s['title']}

**Origin:** {s['author']}  
**Type:** {s['type']}  
**Summary:** {s['summary']}  
**Limitation:** {s['limitation']}
'''

for c in scored_claims:
    planned_writes[Path(f"Claims/{c['id']} - Colab Reproduction.md")] = f'''---
type: claim-reproduction
claim_id: {c['id']}
claim_type: {c['claim_type']}
confidence_score: {c['confidence_score']}
stale: {str(c['stale']).lower()}
synthetic: true
---
# {c['id']} — {c['statement']}

Status: **{c['status']}**. Sources: {', '.join(c['sources'])}.
'''

print("Artifacts in memory:", len(planned_writes))
print("Files written: 0")


In [ ]:
# 12. Preview exact mutations and content hashes

mutation_manifest = []
for relative, content in planned_writes.items():
    target = VAULT / relative
    new_bytes = content.encode("utf-8")
    new_hash = hashlib.sha256(new_bytes).hexdigest()
    action = "CREATE" if not target.exists() else ("UNCHANGED" if sha256(target) == new_hash else "UPDATE")
    mutation_manifest.append({"action":action,"path":str(relative),"bytes":len(new_bytes),"sha256":new_hash})
display_table(mutation_manifest, ["action","path","bytes","sha256"], "Proposed mutations")
print("COMMIT_CHANGES:", COMMIT_CHANGES)


## Human commit gate

Review the source limitations, claim classifications, score components, calculation, contradiction and decision before changing `COMMIT_CHANGES` to `True`. Existing targets are backed up and writes use atomic replacement.


In [ ]:
# 13. Commit only when explicitly authorized

def atomic_write(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=path.name + ".", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f: f.write(content.rstrip() + "\n")
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name): os.unlink(temp_name)

committed = []
if not COMMIT_CHANGES:
    print("DRY RUN COMPLETE — no files were written.")
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup_root = VAULT / "Audit/Backups" / f"{LOOP_ID}-{stamp}"
    for relative, content in planned_writes.items():
        target = VAULT / relative
        if target.exists() and sha256(target) == hashlib.sha256(content.encode()).hexdigest():
            committed.append({"action":"UNCHANGED","path":str(relative)}); continue
        if target.exists() and BACKUP_BEFORE_WRITE:
            backup = backup_root / relative; backup.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(target, backup)
        action = "UPDATE" if target.exists() else "CREATE"
        atomic_write(target, content); committed.append({"action":action,"path":str(relative)})
    display_table(committed, ["action","path"], "Committed changes")


In [ ]:
# 14. Final validation and hot-cache payload

validation = {
    "sources_unique": len({s['id'] for s in SOURCES}) == len(SOURCES),
    "claims_unique": len({c['id'] for c in scored_claims}) == len(scored_claims),
    "all_sources_resolved": all(set(c['sources']) <= {s['id'] for s in SOURCES} for c in scored_claims),
    "derived_growth_recalculated": calculation_check['matches'],
    "contradiction_preserved": contradiction_open,
    "inference_cap_respected": next(c for c in scored_claims if c['id']=='CLM-006')['confidence_score'] <= 70,
    "writes_authorized": COMMIT_CHANGES,
}
display_table([{"Validation":k,"Result":v} for k,v in validation.items()], ["Validation","Result"], "Final validation")
assert all(v for k,v in validation.items() if k != "writes_authorized")

hot_cache_payload = {
    "loop_id": LOOP_ID,
    "company": "SYN-102 — VoltEdge Thermal Systems",
    "sources": len(SOURCES),
    "claims": len(scored_claims),
    "average_confidence": average_confidence,
    "open_contradictions": 1,
    "decision": decision,
    "external_actions": 0,
    "next_test": "Introduce one real public company using dated public sources and the same taxonomy.",
}
print(json.dumps(hot_cache_payload, indent=2, ensure_ascii=False))


## What Baby Step 3 changes

The system no longer stores unsupported conclusions as if they were facts. It now has source identity, source dates, claim lineage, claim types, evidence-quality components, review horizons, contradiction preservation and a decision gate. The result is intentionally conservative: VoltEdge may enter internal preliminary diligence, but the unresolved capacity conflict blocks valuation reliance and external action.
